## Volume drivers & sharing

**Every volume has a driver** — the plugin that actually stores the bytes. The default is **`local`**: a directory on this host at `/var/lib/docker/volumes/<name>/_data`. You can pass options at creation to change *what* `local` points at — most usefully an NFS share:

```bash
docker volume create --driver local \
  --opt type=nfs --opt o=addr=192.168.1.10,rw \
  --opt device=:/exports/data myvol
```

Third-party plugins extend the list to cloud block storage (AWS EBS, GCP PD), distributed filesystems (Ceph, GlusterFS), and object-storage shims. But for a **single-host** install, the default `local` driver is what ~95% of teams use — drivers really start to matter with multi-host orchestrators (Swarm shares state via volume drivers; Kubernetes has its own CSI ecosystem).

**Sharing a volume between containers.** The modern way is simply to **mount the same named volume in each**:

```bash
docker volume create shared
docker run -d --name producer -v shared:/out alpine sh -c 'while true; do date >> /out/log; sleep 1; done'
docker run --rm -v shared:/in alpine cat /in/log
```

The volume is the source of truth; each container mounts it wherever it likes. (An older flag, **`--volumes-from <container>`**, copies *all* of another container's mounts in — the legacy "data container" pattern. Recognise it in old setups; don't reach for it now that volumes are first-class objects.)

**One caution:** volumes give you shared storage, **not** coordinated writes. Two containers writing the same files can corrupt data unless the *application* coordinates — Postgres refuses two servers on one data dir; SQLite struggles with concurrent writers. The volume shares the bytes; safe concurrency is the app's job.